In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

login(token=hf_token)

In [ ]:
from datasets import get_dataset_split_names, load_dataset

DATASET_NAME = "nexar-ai/nexar_collision_prediction"

print("🔍 در حال بررسی تمامی اسپلیت‌ها...\n")

try:
    available_splits = get_dataset_split_names(DATASET_NAME)
    print(f" اسپلیت‌های موجود: {available_splits}\n")
    print("=" * 60)

    for split_name in available_splits:
        print(f"\n بررسی اسپلیت: [{split_name}]")
        ds = load_dataset(DATASET_NAME, split=split_name, streaming=True)
        sample = next(iter(ds))

        print(" فیلدهای متادیتا:")
        for key, val in sample.items():
            if key == 'video':
                print(f"  • {key}: <VideoDecoder>")
            else:
                print(f"  • {key}: {val} ({type(val).__name__})")
        print("-" * 60)

except Exception as e:
    print(f" خطا: {e}")

In [ ]:
import os
import gc
import json
import torch
import torch.nn.functional as F
import numpy as np
from torchvision import models
from datasets import load_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"اجرای فرایند روی: {device}")

resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
resnet.fc = torch.nn.Identity()
resnet = resnet.to(device).eval()

imagenet_mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
imagenet_std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

WINDOW_SEC = 3.0
OUT_DIR = "extracted_features_test"
os.makedirs(OUT_DIR, exist_ok=True)

print(" در حال بارگذاری اسپلیت test...")
ds = load_dataset("nexar-ai/nexar_collision_prediction", split="test", streaming=True)

print(" شروع استخراج ویژگی ۳ ثانیه پایانی ویدیوها...\n")

for idx, sample in enumerate(ds):
    file_id = f"{idx:05d}"
    npy_path = f"{OUT_DIR}/{file_id}.npy"
    json_path = f"{OUT_DIR}/{file_id}.json"

    if os.path.exists(npy_path):
        continue

    try:
        decoder = sample['video']
        
        try:
            fps = float(decoder.metadata.average_fps)
            total_frames = decoder.metadata.num_frames
            duration = float(decoder.metadata.duration_seconds)
        except Exception:
            fps, total_frames = 30.0, len(decoder)
            duration = total_frames / fps

        end_sec = duration
        start_sec = max(0.0, end_sec - WINDOW_SEC)

        start_frame = int(start_sec * fps)
        end_frame = min(total_frames, int(end_sec * fps))

        if end_frame <= start_frame:
            continue

        frame_batch = decoder[start_frame:end_frame]
        frames_tensor = frame_batch.data if hasattr(frame_batch, 'data') else frame_batch
        frames_tensor = frames_tensor.to(device).float() / 255.0

        frames_resized = F.interpolate(frames_tensor, size=(224, 224), mode='bilinear', align_corners=False)
        frames_normalized = (frames_resized - imagenet_mean) / imagenet_std

        with torch.no_grad():
            features = resnet(frames_normalized).cpu().numpy()

        np.save(npy_path, features)

        meta_data = {
            "id": file_id,
            "sample_index": idx,
            "duration_sec": duration,
            "window_start_sec": start_sec,
            "window_end_sec": end_sec,
            "extracted_frames": len(features)
        }
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(meta_data, f, indent=4)

        if (idx + 1) % 50 == 0:
            print(f" [{idx + 1}/1348] ویدیو با موفقیت پردازش شد.", flush=True)

    except Exception as e:
        print(f" خطا در پردازش نمونه {idx}: {str(e)}", flush=True)

    finally:
        if 'frames_tensor' in locals(): del frames_tensor
        if 'frames_resized' in locals(): del frames_resized
        if 'frames_normalized' in locals(): del frames_normalized
        if 'features' in locals(): del features
        if idx % 20 == 0:
            torch.cuda.empty_cache()
            gc.collect()

print("\n استخراج تمامی داده‌های تست به پایان رسید!")